<a href="https://colab.research.google.com/github/AmiraFaisal/ETEC2T/blob/main/BARTScore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
drive.mount('/content/drive', force_remount=True)

# Verify paths exist
qwen_path = "/content/drive/MyDrive/EvaltheEvaluators/Qwen_Model/Qwen_Descriptions.json"
gemma_path = "/content/drive/MyDrive/EvaltheEvaluators/ChartGemma_Model/ChartGemma_Descriptions.json"
images_dir = "/content/drive/MyDrive/EvaltheEvaluators/images"
GT_dir = "/content/drive/MyDrive/EvaltheEvaluators/images/L2L3_captions.json"

# Check Qwen JSON
if os.path.exists(qwen_path):
    with open(qwen_path, 'r') as f:
        qwen = json.load(f)
    print(f"\nJSON loaded: {len(qwen)} samples")
    print(f"Sample: {qwen[0]}")
else:
    print(f"JSON not found: {qwen_path}")

# Check ChartGemma JSON
if os.path.exists(gemma_path):
    with open(gemma_path, 'r') as f:
        gemma = json.load(f)
    print(f"\nJSON loaded: {len(gemma)} samples")
    print(f"Sample: {gemma[0]}")
else:
    print(f"JSON not found: {gemma_path}")

# Check images
if os.path.exists(images_dir):
    all_files = os.listdir(images_dir)
    images = [f for f in all_files if f.lower().endswith(('.png'))]
    print(f"\nImages directory: {len(images)} files")
    print(f"Sample: {images}")
else:
    print(f"Images directory not found: {images_dir}")

# Check ground truths (L2L3_captions.json)
if os.path.exists(GT_dir):
    with open(GT_dir, 'r') as f:
        GT = json.load(f)
    print(f"\nground truths loaded:")
else:
    print(f"JSON not found: {GT_dir}")

Mounted at /content/drive

JSON loaded: 21 samples
Sample: {'img_id': '3128', 'answer': "The graph depicts the infant mortality rate from 2009 to 2018 for Armenia (in deaths per 1,000 live births). The starting point is at approximately 16 deaths per 1,000 live births, which gradually decreases over time until it reaches around 7 by 2018. There's no significant increase or decrease observed during this period. Overall, there seems to have been a steady decline in the infant mortality rate within the given timeframe."}

JSON loaded: 21 samples
Sample: {'img_id': 3128, 'answer': 'The chart shows the infant mortality rate in Armenia from 2009 to 2019. The rate is measured in deaths per 1,000 live births. The chart shows that the rate has been steadily decreasing over the past decade. In 2009, the rate was around 15 deaths per 1,000 live births. By 2019, the rate had fallen to around 10 deaths per 1,000 live births. This represents a significant improvement in the health of infants in Arme

In [1]:
!wget https://raw.githubusercontent.com/neulab/BARTScore/main/bart_score.py
!pip install rouge_score --quiet
import json
import os
import pandas as pd
import numpy as np
import torch
from rouge_score import rouge_scorer
from google.colab import drive
from bart_score import BARTScorer
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

--2026-04-05 09:12:18--  https://raw.githubusercontent.com/neulab/BARTScore/main/bart_score.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4221 (4.1K) [text/plain]
Saving to: ‘bart_score.py’

bart_score.py       100%[===================>]   4.12K  --.-KB/s    in 0s      

2026-04-05 09:12:19 (15.2 MB/s) - ‘bart_score.py’ saved [4221/4221]

  Preparing metadata (setup.py) ... done


In [3]:
# prepare the already loaded data into maps
def createDict(data_list, id_key='img_id', text_key='answer'):
    """Create a dict that maps string-IDs to captions from a list of dicts."""
    return {str(item[id_key]): item[text_key] for item in data_list}

# GT has 'caption_L2L3' as the text key, qwen and gemma have 'answer' from their respective JSONs
human_map = createDict(GT, text_key="caption_L2L3")
qwen_map = createDict(qwen, text_key="answer")
gemma_map = createDict(gemma, text_key="answer")

# align the lists based on the *intersection* of image IDs present in all datasets
# since `caption_L2L3` from `GT` contains for all original charts including the selected 21
common_ids = sorted(list(set(human_map.keys()) & set(qwen_map.keys()) & set(gemma_map.keys())))
references = [human_map[i] for i in common_ids]
candidates_qwen = [qwen_map[i] for i in common_ids]
candidates_gemma = [gemma_map[i] for i in common_ids]

In [4]:
# initialize BARTScorer
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
bart_scorer = BARTScorer(device=device, checkpoint='facebook/bart-large-cnn')

# calculate BARTScores
# higher (less negative) is better. e.g., -1.5 is better than -4.0
bs_qwen = bart_scorer.score(candidates_qwen, references, batch_size=4)
bs_gemma = bart_scorer.score(candidates_gemma, references, batch_size=4)

# final combined results table
BARTScore_results = {
    "Model": ["Qwen", "ChartGemma"],
    "BARTScore": [sum(bs_qwen)/len(bs_qwen), sum(bs_gemma)/len(bs_gemma)]
}

BARTScore_df = pd.DataFrame(BARTScore_results)
print("\n--- BARTScore ---")
print(BARTScore_df.to_string(index=False, float_format=lambda x: "{:.4f}".format(x)))

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]


--- BARTScore ---
     Model  BARTScore
      Qwen    -3.2023
ChartGemma    -3.3072


assuming `bs_qwen` and `bs_gemma` are the lists of scores from `bart_scorer.score()`

In [9]:
# simple Softmax-style Normalization
def get_readable_score(log_score):
    # 'clipping' technique to bring BARTScore into a 1-10 range
    return np.exp(log_score) * 10

# Create per-question DataFrame
per_question_data = {
    "img_id": common_ids,
    "Qwen": [get_readable_score(score) for score in bs_qwen],
    "ChartGemma": [get_readable_score(score) for score in bs_gemma]
}
df_per_question = pd.DataFrame(per_question_data)
print(df_per_question.to_string(index=False, float_format=lambda x: "{:.4f}".format(x)))

img_id   Qwen  ChartGemma
  1061 0.1431      0.1346
  1064 0.0797      0.0823
  1253 0.1191      0.0854
  1356 0.4642      0.6087
  2114 0.1537      0.1367
  2121 0.5325      0.6006
  2264 0.1841      0.1608
  2369 0.8141      0.8138
  2472 0.2919      0.3972
  3128 0.4638      0.2869
  3190 1.7706      2.3725
  4067 0.4712      0.5493
  4845 1.6326      1.7273
  4915 0.4794      0.4952
  5932 0.3386      0.3296
   623 0.5744      0.3905
  7127 0.6480      0.4228
   749 0.3271      0.2909
  7567 0.7911      0.7200
  8038 1.0043      0.2315
  8673 0.2787      0.3317


In [11]:
drive_path = '/content/drive/MyDrive/EvaltheEvaluators/evaluations/Coherence/BARTScore_results.csv'
df_per_question.to_csv(drive_path, index=False)


higher BARTScore (less negative, closer to 0) indicates better quality, meaning the generated text is more coherent, consistent, and fluent compared to reference.